# Data science salaries: model comparison, redone in Python

Redo of the PSTAT 131 R project (`main.Rmd`). The R analysis had an evaluation bug (the linear model's test MSE mixed log and USD scales), unscaled KNN, a single train/test split and duplicate rows leaking across the split. Here every model is scored in USD on held-out folds. Run `model_comparison.py` and `tune_models.py` first to regenerate `outputs/`.

In [1]:
import json
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pandas as pd
from model_comparison import load, CAT

pd.options.display.float_format = '{:,.3f}'.format
raw, dedup = load(False), load(True)
print(f'{len(raw)} rows, {len(raw)-len(dedup)} exact duplicates ({1-len(dedup)/len(raw):.0%}) -> {len(dedup)} unique')

3755 rows, 1172 exact duplicates (31%) -> 2583 unique


## 1. The target and the data

In [2]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
dedup.salary_in_usd.plot.hist(bins=40, ax=ax[0], title='salary_in_usd'); 
dedup.groupby('experience_level').salary_in_usd.median().reindex(['EN','MI','SE','EX']).plot.bar(ax=ax[1], title='Median salary by experience')
dedup.groupby('company_location').salary_in_usd.agg(['median','count']).query('count>=20')['median'].sort_values().plot.barh(ax=ax[2], title='Median salary, locations with >=20 rows')
plt.tight_layout()

Most rows are US companies, so location effects are dominated by the US versus the rest. Many job titles and locations have only a handful of rows, which limits what any model can learn.

## 2. Duplicates inflate every score

In [3]:
res = pd.read_csv('outputs/cv_results.csv')
piv = res.pivot(index='model', columns='data', values='rmse')
piv['inflation'] = piv['deduplicated'] - piv['with dupes']
piv.sort_values('deduplicated').round(0)

data,deduplicated,with dupes,inflation
model,,,
Ridge (log target),"51,175.000","48,888.000","2,287.000"
Random forest,"51,577.000","48,248.000","3,329.000"
XGBoost (log target),"51,632.000","48,908.000","2,724.000"
LightGBM (log target),"52,254.000","49,264.000","2,990.000"
KNN k=15 (scaled),"55,639.000","51,949.000","3,690.000"
Median baseline,"67,224.000","63,083.000","4,141.000"


Every model looks roughly $2,300 to $4,100 better when duplicates stay in, because copies of the same row land in both train and test folds. All comparisons below use the deduplicated data.

## 3. Model comparison (repeated 5-fold CV, 3 repeats, deduplicated)

In [4]:
d = res[res.data=='deduplicated'].set_index('model').sort_values('rmse')
display(d[['rmse','rmse_sd','mae','r2']].round(3))
d['rmse'].plot.barh(xerr=d['rmse_sd'], figsize=(8,3.5), title='CV RMSE (USD), lower is better'); plt.gca().invert_yaxis()

,rmse,rmse_sd,mae,r2
model,,,,
Ridge (log target),"51,174.677","3,172.263","36,666.110",0.416
Random forest,"51,577.205","2,942.262","38,004.502",0.407
XGBoost (log target),"51,632.356","3,102.313","36,957.257",0.405
LightGBM (log target),"52,253.835","3,193.825","37,599.219",0.391
KNN k=15 (scaled),"55,639.393","3,179.596","41,953.520",0.310
Median baseline,"67,224.093","2,904.889","52,868.005",-0.007


Ridge on a log target is as good as anything else. The R report's claim that tree methods beat linear regression "by two digits" came from comparing log-scale predictions with dollars. With the scale fixed, they are within about 1% of each other, and scaled KNN is a reasonable baseline rather than a failure.

## 4. Does tuning help? (nested CV)

In [5]:
tuned = pd.read_csv('outputs/tuned_results.csv').set_index('model')
base = d.loc[['Random forest','XGBoost (log target)','LightGBM (log target)'], ['rmse','r2']]
base.index = [i + ' tuned' for i in base.index]
cmp = base.join(tuned[['rmse','r2']], lsuffix=' default', rsuffix=' tuned')
cmp.columns = ['rmse default','r2 default','rmse tuned','r2 tuned']
cmp.round(3)

,rmse default,r2 default,rmse tuned,r2 tuned
Random forest tuned,"51,577.205",0.407,"51,535.667",0.409
XGBoost (log target) tuned,"51,632.356",0.405,"51,994.579",0.398
LightGBM (log target) tuned,"52,253.835",0.391,"52,053.619",0.396


Tuning with randomized search inside each outer fold changes RMSE by well under 1%, which is inside the fold-to-fold noise. The limit is the information in seven features, not the hyperparameters.

## 5. What drives salary

In [6]:
plt.figure(figsize=(8,7)); plt.imshow(mpimg.imread('outputs/shap_summary.png')); plt.axis('off')

(np.float64(-0.5), np.float64(1199.5), np.float64(1124.5), np.float64(-0.5))

A US company location is the largest single effect on log-salary. Experience level comes next: entry level pulls salary down and senior and executive levels push it up. An Indian company location and a Data Analyst title lower it, and Machine Learning Engineer and later work years raise it. These are associations in a survey sample, not causal effects.

## 6. Prediction intervals

In [7]:
cov = json.load(open('outputs/interval_coverage.json'))
print(f"Nominal {cov['nominal']:.0%} interval: out-of-fold coverage {cov['coverage']:.1%}, median width ${cov['median_width']:,.0f}")

Nominal 80% interval: out-of-fold coverage 77.5%, median width $115,245


LightGBM quantile models give an 80% interval that covers about 77% of held-out salaries, close to nominal, but the median interval is over $100k wide. That is the honest picture for a point estimate with R² around 0.4, and it is why the planned web app should show a range, not a single number.

## Next steps
- Build the estimator and explorer UI on a quantile model (the interval is the product).
- Collect or join more informative features (company industry, skills, years of experience) since tuning can't fix missing signal.
- Check calibration of the interval per segment (US vs non-US, entry vs senior).